In [ ]:
import pandas as pd
import pickle
import numpy as np

# Load the trained model
with open('Model/smart_admission_model.pkl', 'rb') as f:
    model = pickle.load(f)


In [23]:
# Example input (simulate new applicant)
applicant_data = {
    'faculty': 'Science',
    'department': 'Computer Science',
    'utme_score': 280,
    'screening_score': 65.0,
    'olevel_passed': True,
    'olevel_grades': ['A1', 'A1', 'B3', 'C2', 'A1']
}

# Grade conversion dictionary
grade_points = {
    'A1': 10, 'B1': 8, 'B2': 7, 'B3': 6,
    'C1': 5, 'C2': 4, 'C3': 3, 'C4': 2,
    'C5': 1, 'C6': 0
}

# Calculate O'Level Score
olevel_score = sum([grade_points.get(grade, 0) for grade in applicant_data['olevel_grades']])

# Screening score = 50% UTME + 50% O'Level (same logic used in training)
utme_component = applicant_data['utme_score'] / 8
screening_score = utme_component + olevel_score

# Determine O’Level pass validity
olevel_passed = all(grade_points.get(g, -1) >= 0 for g in applicant_data['olevel_grades'])

# Create dataframe
input_df = pd.DataFrame([{
    'faculty': applicant_data['faculty'],
    'department': applicant_data['department'],
    'utme_score': applicant_data['utme_score'],
    'screening_score': screening_score,
    'olevel_passed': olevel_passed,
    'olevel_grades': str(applicant_data['olevel_grades'])
}])

# Clean up applicant_data input
applicant_data['faculty'] = applicant_data['faculty'].replace('Faculty of ', '').strip()

# Encode 'faculty' and 'department' with same encoders used during training
# Encode using previously trained LabelEncoders
input_df['faculty_encoded'] = faculty_encoder.transform(input_df['faculty'])
input_df['department_encoded'] = department_encoder.transform(input_df['department'])

# Add scaled scores
input_df['utme_scaled'] = input_df['utme_score'] / 400
input_df['screening_scaled'] = input_df['screening_score'] / 100

# Drop original categorical columns
input_df.drop(columns=['faculty', 'department', 'olevel_grades'], inplace=True)

# Reorder or select the exact features used in training
input_df = input_df[['faculty_encoded', 'department_encoded', 'utme_score', 'utme_scaled', 'screening_score', 'screening_scaled', 'olevel_passed']]

# For now assume model was trained with original string categories

print("Valid Faculties:", list(faculty_encoder.classes_))
print("Valid Departments:", list(department_encoder.classes_))

import joblib

# Load the trained model
model = joblib.load('Model/smart_admission_model.pkl')

# Load the saved encoders
model = joblib.load('Model/smart_admission_model.pkl')
faculty_encoder = joblib.load('Model/faculty_encoder.pkl')
department_encoder = joblib.load('Model/department_encoder.pkl')

# Predict
prediction = model.predict(input_df)[0]
probability = model.predict_proba(input_df)[0][1]

# Show result
result = "Admitted ✅" if prediction == 1 else "Not Admitted ❌"
print(f"\n📍 Prediction: {result}")
print(f"🎯 Admission Probability: {round(probability * 100, 2)}%")


Valid Faculties: ['Arts', 'Clinical Science', 'Engineering', 'Law', 'Management Sciences', 'Science', 'Social Sciences']
Valid Departments: ['Accounting', 'Aeronautical and Astronautical Engineering', 'Banking and Finance', 'Botany', 'Business Administration', 'Chemical and Polymer Engineering', 'Chemistry', 'Civil Engineering', 'Commercial and Industrial Law', 'Computer Science', 'Creative Arts: Music', 'Creative Arts: Theatre', 'Creative Arts: Visual', 'Criminology', 'Dentistry', 'Economics', 'Electronic and Computer Engineering', 'English Language', 'English Literature', 'Fisheries', 'Foreign Languages', 'Geography and Planning', 'History and International studies', 'Industrial Engineering', 'Industrial Relations', 'Insurance', 'Islamic Law', 'Jurisprudence and International Law', 'Legal Studies', 'Linguistics', 'Marketing', 'Mathematics', 'Mechanical Engineering', 'Medicine and Surgery', 'Microbiology', 'Nursing', 'Pharmacology', 'Philosophy', 'Physics', 'Political Science', 'Proje